In [73]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 

In [74]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [75]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=8):
        
        # one_hot_encoded = np.zeros((len(data), 1), dtype=float)
        # for ii, token in enumerate(data):
        #     one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                self.X[ii,jj] = \
                ord(data[ii+jj])-65
                    
            self.y[ii] = \
                ord(data[ii+jj+1])-65

        self.X = tnsr(self.X).long()
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [76]:
class base_model(nn.Module):
    def __init__(self, input_size, embedding_dim, hidden_size, hidden_sleep_size, context_size, output_size=7, num_layers=1, modulation_strength=0):
        super().__init__()

        self.context_fc = nn.Linear(hidden_sleep_size, context_size, bias=False)
        self.embedding = nn.Embedding(input_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim+context_size, hidden_size, num_layers, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.modulation_strength = modulation_strength

    def set_modulation(self, val):
        self.modulation_strength = val
        
    def forward(self, x, hs, h=None):
        context = self.context_fc(hs)
        x = self.embedding(x)

        # print(x, context)
        x = torch.cat(((1-self.modulation_strength)*x, self.modulation_strength*context), dim=2)
        
        out, h = self.rnn(x, h)
        out = self.fc(out[:,-1,:])

        return out, h
    

In [94]:
class RNNEncoder(nn.Module):
    def __init__(self, input_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Linear(input_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
    
    def forward(self, x, h=None):
        embedded = self.embedding(x)
        out, h = self.rnn(embedded, h)
        return h  
    
class RNNDecoder(nn.Module):
    def __init__(self, input_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Linear(input_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, input_size)
    
    def forward(self, x, h):
        embedded = self.embedding(x)
        out, _ = self.rnn(embedded, h)
        return self.fc(out) 
    
#########################################
class context_layer(nn.Module):
    def __init__(self, input_size, embedding_dim, hidden_size):
        super().__init__()
        self.encoder = RNNEncoder(input_size, embedding_dim, hidden_size)
        self.decoder = RNNDecoder(input_size, embedding_dim, hidden_size)
    
    def forward(self, x, decoder_input, h=None):
        h = self.encoder(x, h)
        output = self.decoder(decoder_input, h)
        return output, h

In [ ]:
class brain:
    def __init__(self, input_size, embedding_dim, hidden_wake_size, hidden_sleep_size, context_output_size, num_layers_wake, num_layers_sleep, output_size=7, wake_lr=1e-3, sleep_lr=1e-3):
        self.wake_model = base_model(input_size, embedding_dim, hidden_wake_size, hidden_sleep_size, context_size=context_output_size, output_size=output_size, num_layers=num_layers_wake)
        self.wake_optimizer = torch.optim.SGD(self.wake_model.parameters(), lr=wake_lr, momentum=0.95)
        self.wake_criterion = torch.nn.CrossEntropyLoss()

        self.sleep_model = context_layer(hidden_wake_size, embedding_dim, hidden_sleep_size)
        self.sleep_optimizer = torch.optim.SGD(self.sleep_model.parameters(), lr=sleep_lr, momentum=0.95)
        self.sleep_criterion = torch.nn.MSELoss()

        self.hidden_wake_size = hidden_wake_size
        self.hidden_sleep_size = hidden_sleep_size

    def wake(self, train_loader, cortical_strength=0.5):
        self.wake_model.set_modulation(cortical_strength)
        total = 0
        test_acc = []
        correct = np.zeros(1000,dtype=float)
        hs = torch.zeros((1,1,self.hidden_sleep_size))
        
        for X, y in train_loader:
            self.wake_optimizer.zero_grad()

            if total == 0:
                predicted_y, h = self.wake_model(X,hs)
            else:
                predicted_y, h = self.wake_model(X, hs, h=mem)
            
            # print(predicted_y, y)
            loss = self.wake_criterion(predicted_y, y[0])
            
            
            loss.backward(retain_graph=True)
            self.wake_optimizer.step()

            with torch.no_grad():
                if total>0 and cosine(mem[0][0], h[0][0]) > 0.6:
                        hs = self.sleep_model.encoder(h.clone(), hs)

                true_y = y[0]
                estimated_y = predicted_y.argmax(axis=1)

                mem = h.clone()
                total += 1
                if true_y == estimated_y:
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0

                test_acc.append(
                    np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                )
                if total%1000 == 0:
                    print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')
            
        return test_acc
    

    def sleep(self, sleep_duration=400000, cortical_strength=0.5, threshold=0.6, context_span=6):
        self.wake_model.set_modulation(cortical_strength)
        X_hat = torch.randint(0, len(tokens), (1,)) [0]
        X_hat = X_hat.view(1,1)
        # threshold = 0.6
        
        # seq = ''
        train_rnn2 = False

        hs = torch.zeros((1,1,self.hidden_sleep_size))

        # prev_h = torch.randn((1,1,hidden_wake_size))
        target_h = torch.zeros((1,context_span,self.hidden_wake_size))
        decoder_input = torch.zeros((1,context_span,self.hidden_wake_size))
        # input_h = torch.zeros((1,context_span,self.hidden_wake_size))

        # mem = hs.clone()
        # count = 0
        for jj in range(sleep_duration):
            # fix_seeds()
            with torch.no_grad():
                if jj == 0:
                    X_hat_, hidden_state = self.wake_model(X_hat, hs)
                else:
                    # hidden_state += SWR*torch.randn(hidden_state.size())
                    X_hat_, hidden_state = self.wake_model(X_hat, hs, hidden_state)
                    
                    if cosine(prev[0][0], hidden_state[0][0]) < threshold:
                        train_rnn2 = False
                        # h_start = hidden_state.clone()
                        # count += 1
                    else:
                        # print('train', cosine(prev[0][0], hidden_state[0][0]), tokens[X_hat.argmax()])
                        # count += 1
                        # seq = seq + tokens[idx]
                        train_rnn2 = True 
                        h_start = hidden_state.clone()
                        target_h[0][1:] = target_h[0][:-1].clone()
                        target_h[0][0] = h_start.clone()
                        decoder_input[0][1:] = target_h[0][:-1]
                        #input_h = target_h.clone()#.reshape(1,total_states_to_predict,hidden_wake_size)
                        

            ### train RNN2 ###
            if train_rnn2:
                self.sleep_optimizer.zero_grad()
                predicted_h, hs = self.sleep_model(target_h, decoder_input)
                loss = self.sleep_criterion(predicted_h, target_h)
                
                
                loss.backward(retain_graph=True)
                self.sleep_optimizer.step()

            with torch.no_grad():
                prev = hidden_state.clone()
                X_hat_prob = torch.nn.functional.softmax(X_hat_, dim=1)
                # fix_seeds(10)
                dist_categ = torch.distributions.Categorical(probs=X_hat_prob.reshape(-1))
                idx = dist_categ.sample()
                # idx = X_hat_prob.argmax()

                # noise = 10*torch.randn(context_output_size)
                # X_hat = torch.zeros(len(tokens),dtype=torch.float32)
                # X_hat[len(tokens):] = noise
                X_hat = idx#[idx] = 1.0
                X_hat = X_hat.reshape(1,-1)
                # prev_h = hidden_state.clone()
                # print(X_hat)
                if jj%10000==0 and jj>1000:
                    print(loss)

        # print(seq)

        
    

In [118]:
# total_samples = 200000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 40
embedding_dim = 2
hidden_sleep_size = 100
context_output_size = 10
num_layers_wake = 1
num_layers_sleep = 1
output_size = len(tokens)
input_size = len(tokens)*working_memory
lr = 1e-3
test_acc = []
wake_sleep_cycle = 3



model = brain(input_size, embedding_dim, hidden_wake_size, hidden_sleep_size, context_output_size, num_layers_wake, num_layers_sleep, output_size=output_size, wake_lr=lr, sleep_lr=lr)

for cycle in range(wake_sleep_cycle):
    if cycle == 0:
        total_samples = 40000
    else:
        total_samples = 400000

    data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

    data_set = Dataset_converter(data, working_memory, short_term_memory)
    train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

    if cycle == 0:
        model.wake(train_loader, cortical_strength=0)
    else:
        model.wake(train_loader)

    model.sleep()

Iter : 1001, loss: 2.4509, accuracy: 0.2760
Iter : 2001, loss: 2.0783, accuracy: 0.4660
Iter : 3001, loss: 1.2846, accuracy: 0.5500
Iter : 4001, loss: 2.1821, accuracy: 0.6120
Iter : 5001, loss: 1.8835, accuracy: 0.5960
Iter : 6001, loss: 1.7612, accuracy: 0.6420
Iter : 7001, loss: 1.7071, accuracy: 0.6200
Iter : 8001, loss: 2.0250, accuracy: 0.6300
Iter : 9001, loss: 1.6328, accuracy: 0.6200
Iter : 10001, loss: 1.7870, accuracy: 0.6280
Iter : 11001, loss: 1.8353, accuracy: 0.6250
Iter : 12001, loss: 2.1079, accuracy: 0.6350
Iter : 13001, loss: 2.1757, accuracy: 0.6160
Iter : 14001, loss: 1.8190, accuracy: 0.6190
Iter : 15001, loss: 1.7234, accuracy: 0.6470
Iter : 16001, loss: 1.6300, accuracy: 0.6760
Iter : 17001, loss: 1.6198, accuracy: 0.6670
Iter : 18001, loss: 1.7843, accuracy: 0.6660
Iter : 19001, loss: 1.7673, accuracy: 0.6580
Iter : 20001, loss: 2.0563, accuracy: 0.6690
Iter : 21001, loss: 2.0367, accuracy: 0.6580
Iter : 22001, loss: 1.7608, accuracy: 0.6650
Iter : 23001, loss:

In [83]:
X

tensor([[1]])

In [6]:
### initial training ###
total_samples = 30000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 40
hidden_sleep_size = 100
context_output_size = 10
num_layers_wake = 1
num_layers_sleep = 1
output_size = len(tokens)
input_size = len(tokens)*working_memory
embedding_dim = 3
lr = 1e-3
test_acc = []
c = []
cortical_strength = 0

data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)#gen_seq(total_samples) #

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = base_model(input_size, embedding_dim, hidden_wake_size, hidden_sleep_size, context_size=context_output_size, output_size=output_size, num_layers=num_layers_wake)
network1.set_modulation(cortical_strength)

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
hs = torch.zeros((1,1,hidden_sleep_size))
for X, y in train_loader:
    # context = torch.zeros((1,X.size(1),context_output_size))
    # X = torch.cat((X,context), dim=2)
    optimizer.zero_grad()

    if total == 0:
        predicted_y, h = network1(X, hs)
    else:
        predicted_y, h = network1(X, hs, h=mem)
    
    loss = criterion(predicted_y, y[0])
    
    
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem = h.clone()
        true_y = y[0]#.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')


Iter : 1001, loss: 1.9450, accuracy: 0.4240
Iter : 2001, loss: 1.6503, accuracy: 0.5840
Iter : 3001, loss: 1.4894, accuracy: 0.6510
Iter : 4001, loss: 1.9391, accuracy: 0.6770
Iter : 5001, loss: 2.0968, accuracy: 0.6660
Iter : 6001, loss: 1.7733, accuracy: 0.6570
Iter : 7001, loss: 2.1014, accuracy: 0.6650
Iter : 8001, loss: 1.9126, accuracy: 0.6800
Iter : 9001, loss: 2.1475, accuracy: 0.6690
Iter : 10001, loss: 1.7581, accuracy: 0.6730
Iter : 11001, loss: 1.7892, accuracy: 0.6490
Iter : 12001, loss: 1.8537, accuracy: 0.6700
Iter : 13001, loss: 1.7132, accuracy: 0.6710
Iter : 14001, loss: 1.6445, accuracy: 0.6820
Iter : 15001, loss: 2.0796, accuracy: 0.6500
Iter : 16001, loss: 1.9021, accuracy: 0.6830
Iter : 17001, loss: 1.9021, accuracy: 0.6640
Iter : 18001, loss: 2.0543, accuracy: 0.6920
Iter : 19001, loss: 1.7200, accuracy: 0.6600
Iter : 20001, loss: 1.9550, accuracy: 0.6530
Iter : 21001, loss: 1.5574, accuracy: 0.6700
Iter : 22001, loss: 1.7149, accuracy: 0.6620
Iter : 23001, loss:

In [64]:
### initial training ###
total_samples = 300
working_memory = 1
short_term_memory = 1
hidden_wake_size = 40
hidden_sleep_size = 100
context_output_size = 10
num_layers_wake = 1
num_layers_sleep = 1
output_size = len(tokens)
input_size = len(tokens)*working_memory
embedding_dim = 3
lr = 1e-3
test_acc = []
c = []
cortical_strength = 0

data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)#gen_seq(total_samples) #

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

# network1 = base_model(input_size+context_output_size, embedding_dim, hidden_wake_size, hidden_sleep_size, context_size=context_output_size, output_size=output_size, num_layers=num_layers_wake)
# network1.set_modulation(cortical_strength)

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
hs = torch.zeros((1,1,hidden_sleep_size))
for X, y in train_loader:
    # context = torch.zeros((1,X.size(1),context_output_size))
    # X = torch.cat((X,context), dim=2)
    # optimizer.zero_grad()

    if total == 0:
        predicted_y, h = network1(X, hs)
    else:
        predicted_y, h = network1(X, hs, h=mem)
        print('train', cosine(h[0][0].detach().numpy(), mem[0][0].detach().numpy()), X)
    
    # loss = criterion(predicted_y, y[0])
    
    
    # loss.backward(retain_graph=True)
    # optimizer.step()

    with torch.no_grad():
        mem = h.clone()
        true_y = y[0]#.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')


train 0.1883522229852168 tensor([[2]])
train 0.3703595136462118 tensor([[0]])
train 1.0 tensor([[6]])
train 1.0 tensor([[5]])
train 0.3500300625184557 tensor([[3]])
train 0.26558769506147717 tensor([[4]])
train 1.0 tensor([[6]])
train 1.0 tensor([[3]])
train 0.37667507428204106 tensor([[4]])
train 0.41674627727138513 tensor([[5]])
train 1.0 tensor([[6]])
train 1.0 tensor([[0]])
train 0.2690198969667653 tensor([[2]])
train 0.25188276298838164 tensor([[1]])
train 1.0 tensor([[6]])
train 1.0 tensor([[5]])
train 0.34750174988278604 tensor([[3]])
train 0.23824880896567358 tensor([[4]])
train 0.9917304777854176 tensor([[6]])
train 1.0 tensor([[3]])
train 0.375773924714602 tensor([[4]])
train 0.41645876872362153 tensor([[5]])
train 1.0 tensor([[6]])
train 1.0 tensor([[3]])
train 0.3745118754458321 tensor([[5]])
train 0.34447817372854916 tensor([[4]])
train 0.8828218938341876 tensor([[6]])
train 1.0 tensor([[3]])
train 0.36794421200563276 tensor([[5]])
train 0.33804878077268785 tensor([[4]])
t

In [ ]:
h

tensor([[[2.3761, 0.0000, 0.0272, 0.4746, 0.0000, 0.0000, 0.0000, 0.4084,
          0.0000, 0.0000, 0.0000, 2.8267, 0.8166, 0.9286, 0.0517, 0.0000,
          1.1356, 0.0000, 0.0000, 0.0000, 0.0000, 1.0785, 0.0992, 0.0000,
          0.7391, 0.0000, 2.6832, 2.7007, 0.0000, 0.0000, 0.0000, 0.9944,
          0.0000, 0.0000, 1.6391, 1.7243, 2.4510, 2.1966, 1.8075, 0.0000]]],
       grad_fn=<StackBackward0>)